In [10]:
# Instalação e configuração no Google Colab
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium beautifulsoup4 > /dev/null

# Imports
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Configuração do navegador
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

# Variáveis
limite_produtos = 120
links = []
pagina = 1

# Loop de coleta por página
while len(links) < limite_produtos:
    url = f"https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page={pagina}"
    print(f"🔎 Acessando página {pagina}: {url}")
    driver.get(url)
    time.sleep(3)

    produtos = driver.find_elements(By.CSS_SELECTOR, "div.vtex-search-result-3-x-galleryItem a")

    if not produtos:
        print("🚫 Nenhum produto encontrado nesta página. Encerrando.")
        break

    for p in produtos:
        href = p.get_attribute("href")
        if href and href not in links:
            links.append(href)

    print(f"➡️ Total de links coletados: {len(links)}")
    pagina += 1

# Limitar ao número máximo desejado
links = links[:limite_produtos]
driver.quit()

# Função para extrair detalhes de produto
def extrair_detalhes_produto(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return None
        soup = BeautifulSoup(response.text, "html.parser")

        nome = soup.find("h1", class_="vtex-store-components-3-x-productNameContainer")
        nome = nome.text.strip() if nome else "N/A"

        codigo = soup.find("span", class_="vtex-product-identifier-0-x-product-identifier__value")
        codigo = codigo.text.strip() if codigo else "0"

        preco_original = soup.select_one(".vtex-product-price-1-x-listPriceWithUnitMultiplier strike")
        preco_original = preco_original.text.strip() if preco_original else "N/A"

        preco_promocional = soup.select_one(".telhanorte-telha-store-app-1-x-preco-best-un")
        preco_promocional = preco_promocional.text.strip() if preco_promocional else "N/A"

        preco_m2 = soup.select_one(".telhanorte-telha-store-app-1-x-preco-m2")
        preco_m2 = preco_m2.text.strip() if preco_m2 else "N/A"

        imagem = soup.find("img", class_="telhanorte-telha-store-app-1-x-imagem-destaque")
        imagem = imagem['src'] if imagem else "N/A"

        return [nome, codigo, preco_original, preco_promocional, preco_m2, imagem, url]
    except Exception as e:
        print(f"Erro ao processar {url}: {e}")
        return None

# Coleta dos dados
dados_produtos = []

for i, link in enumerate(links):
    print(f"[{i+1}/{len(links)}] Coletando: {link}")
    detalhes = extrair_detalhes_produto(link)
    if detalhes:
        dados_produtos.append(detalhes)

# Salvar como CSV
df = pd.DataFrame(dados_produtos, columns=[
    "Nome", "Código", "Preço Original", "Preço Promocional", "Preço por m²", "Imagem", "URL"
])
df.to_csv("telhanorte_porcelanatos_detalhado.csv", index=False)
df.head()


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔎 Acessando página 1: https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page=1
➡️ Total de links coletados: 40
🔎 Acessando página 2: https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page=2
➡️ Total de links coletados: 80
🔎 Acessando página 3: https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page=3
➡️ Total de links coletados: 120
[1/120] Coletando: https://www.telhanorte.com.br/porcelanato-nord-ris-90x90-nat-ret-cx-2-42-m2-portobello-2547910/p
[2/120] Coletando: https://www.telhanorte.com.br/porcelanato-confete-white-natural-retificado-100x100cm-caixa-c-2m2-ceusa-1751271/p
[3/120] Coletando: https://www.telhanorte.com.br/porcelanato-mont-blanc-acetinado-90x90cm-r

,Nome,Código,Preço Original,Preço Promocional,Preço por m²,Imagem,URL
0,Porcelanato Nord Ris 90X90Cm Natural Retificad...,2547910,N/A,"R$ 250,00","R$ 103,31/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-nord...
1,Porcelanato Confete White Natural 100x100cm Re...,1751271,N/A,"R$ 300,61","R$ 150,31/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-conf...
2,Porcelanato Mont Blanc Acetinado 90X90Cm Ret C...,2518449,N/A,"R$ 181,28","R$ 111,90/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-mont...
3,Porcelanato Cimentício Acetinado Borda Reta In...,2363550,N/A,"R$ 130,08","R$ 59,13/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-esma...
4,"Porcelanato Caminho Sgr Natural 87,7X87,7Cm Re...",2515032,N/A,"R$ 246,25","R$ 159,90/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-cami...
